# F1-scientific-python — Practice p24 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** seaborn-programming, random-seeding, aggregation-axis

**Set C — integration and challenge · Budget: 40 minutes.**

A teammate claims the revised process has higher scores, but the comparison below is not reproducible or auditable:

```python
rng = np.random.default_rng()
baseline = rng.normal(70, 8, 200)
revised = rng.normal(74, 8, 200)
sns.histplot(x=baseline)
sns.histplot(x=revised)
plt.axvline(revised.mean())
plt.show()
```

Complete all three parts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

## Part A — diagnosis

In 5–8 sentences, identify at least six concrete defects. Your diagnosis must cover reproducibility, comparable binning, axes ownership, labels/legend, overlap visibility, and the ambiguous mean marker.

*Write your diagnosis here.*

The generator is unseeded, so its samples and the resulting picture change from run to run. The two histogram calls choose bins independently, so their bar heights do not describe the same score intervals. The calls also rely on implicit current-axes state instead of creating and passing an owned `Axes`. Neither group is labeled and there is no legend, so a reader cannot reliably identify the layers. Default opaque overlap can hide one histogram behind the other, so explicit transparency is needed. The title and axis labels are missing, leaving the quantity and units unstated. Finally, the lone unlabeled vertical line marks only the revised mean, making both its identity and the baseline comparison ambiguous.

## Part B — reproducible repair

Implement exactly `def build_comparison():`. It must:

- define `SEED = 20260804`, create one seeded generator, and draw `baseline` first and `revised` second with the same parameters shown above;
- define `shared_edges = np.linspace(35.0, 100.0, 14)`, which encloses every value in both fixed seeded samples;
- create one explicit `fig, ax = plt.subplots(figsize=(7, 4))`;
- make exactly two `sns.histplot` calls on that axes, both using `bins=shared_edges`, `alpha=0.45`, and labels `baseline` and `revised`;
- set exact title `Baseline and revised score distributions`, x-label `score`, y-label `count`, and add a legend;
- add two mean markers with `ax.axvline`, dashed line style, and labels `baseline mean` and `revised mean`;
- return `(fig, ax, baseline, revised, shared_edges)` without calling `plt.show()`.

**Banned (zero points): unseeded randomness, separate bin-edge arrays, implicit current-axes plotting, and `plt.hist`.**

In [ ]:
def build_comparison():
    SEED = 20260804
    rng = np.random.default_rng(SEED)
    baseline = rng.normal(70, 8, 200)
    revised = rng.normal(74, 8, 200)
    shared_edges = np.linspace(35.0, 100.0, 14)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(x=baseline, bins=shared_edges, alpha=0.45,
                 label="baseline", ax=ax)
    sns.histplot(x=revised, bins=shared_edges, alpha=0.45,
                 label="revised", ax=ax)
    ax.axvline(baseline.mean(), linestyle="--", label="baseline mean")
    ax.axvline(revised.mean(), linestyle="--", label="revised mean")
    ax.set_title("Baseline and revised score distributions")
    ax.set_xlabel("score")
    ax.set_ylabel("count")
    ax.legend()
    return fig, ax, baseline, revised, shared_edges

Create the required summary identifiers, then run the immutable contract check below:

- `mean_baseline` and `mean_revised` — Python floats;
- `mean_shift` — `mean_revised - mean_baseline`, a Python float.

Do not edit the check: it independently recomputes both numerical histograms and reads both groups' patch heights from the axes.

In [ ]:
histplot_calls = []
_original_histplot = sns.histplot
_original_show = plt.show

def _record_histplot(*args, **kwargs):
    histplot_calls.append((args, kwargs.copy()))
    return _original_histplot(*args, **kwargs)

def _forbid_show(*args, **kwargs):
    raise AssertionError("build_comparison must not call plt.show()")

sns.histplot = _record_histplot
plt.show = _forbid_show
try:
    fig, ax, baseline, revised, shared_edges = build_comparison()
finally:
    sns.histplot = _original_histplot
    plt.show = _original_show

mean_baseline = float(baseline.mean())
mean_revised = float(revised.mean())
mean_shift = float(mean_revised - mean_baseline)

expected_rng = np.random.default_rng(20260804)
expected_baseline = expected_rng.normal(70, 8, 200)
expected_revised = expected_rng.normal(74, 8, 200)
expected_edges = np.linspace(35.0, 100.0, 14)

assert isinstance(mean_baseline, float) and isinstance(mean_revised, float)
assert isinstance(mean_shift, float)
assert np.array_equal(baseline, expected_baseline)
assert np.array_equal(revised, expected_revised)
assert np.array_equal(shared_edges, expected_edges)
assert shared_edges[0] <= min(baseline.min(), revised.min())
assert shared_edges[-1] >= max(baseline.max(), revised.max())
assert len(histplot_calls) == 2
for (call_args, call_kwargs), values, label in zip(
        histplot_calls, (baseline, revised), ("baseline", "revised")):
    assert call_args == ()
    required_kwargs = {"x", "bins", "alpha", "label", "ax"}
    assert required_kwargs <= set(call_kwargs)
    assert call_kwargs["x"] is values
    assert call_kwargs["bins"] is shared_edges
    assert call_kwargs["alpha"] == 0.45
    assert call_kwargs["label"] == label
    assert call_kwargs["ax"] is ax
assert ax.get_title() == "Baseline and revised score distributions"
assert ax.get_xlabel() == "score" and ax.get_ylabel() == "count"
assert np.allclose(fig.get_size_inches(), (7, 4), atol=1e-12, rtol=0)
assert np.allclose(mean_baseline, baseline.mean(), atol=1e-12, rtol=0)
assert np.allclose(mean_revised, revised.mean(), atol=1e-12, rtol=0)
assert np.allclose(mean_shift, mean_revised - mean_baseline, atol=1e-12, rtol=0)
expected_baseline_counts, _ = np.histogram(baseline, bins=shared_edges)
expected_revised_counts, _ = np.histogram(revised, bins=shared_edges)
patch_heights = np.array([patch.get_height() for patch in ax.patches])
n_bins = len(shared_edges) - 1
assert patch_heights.shape == (2 * n_bins,)
drawn_baseline_counts = patch_heights[:n_bins]
drawn_revised_counts = patch_heights[n_bins:]

def _effective_alpha(artist):
    explicit_alpha = artist.get_alpha()
    if explicit_alpha is not None:
        return float(explicit_alpha)
    color_alphas = []
    for accessor in ("get_facecolor", "get_edgecolor"):
        if hasattr(artist, accessor):
            colors = np.asarray(getattr(artist, accessor)())
            if colors.size:
                color_alphas.extend(np.atleast_2d(colors)[:, -1].tolist())
    if hasattr(artist, "get_color"):
        color_alphas.append(to_rgba(artist.get_color())[3])
    return max(color_alphas, default=1.0)

assert np.array_equal(drawn_baseline_counts, expected_baseline_counts)
assert np.array_equal(drawn_revised_counts, expected_revised_counts)
assert expected_baseline_counts.sum() == len(baseline)
assert expected_revised_counts.sum() == len(revised)
for group_patches in (ax.patches[:n_bins], ax.patches[n_bins:]):
    assert all(patch.get_visible() and _effective_alpha(patch) > 0 for patch in group_patches)
    assert np.allclose([patch.get_x() for patch in group_patches], shared_edges[:-1], atol=1e-12, rtol=0)
    assert np.allclose([patch.get_width() for patch in group_patches], np.diff(shared_edges), atol=1e-12, rtol=0)
    assert np.allclose([patch.get_facecolor()[3] for patch in group_patches], 0.45, atol=1e-12, rtol=0)
assert len(ax.lines) == 2
mean_lines = {line.get_label(): line for line in ax.lines}
assert set(mean_lines) == {"baseline mean", "revised mean"}
for label, expected_mean in (("baseline mean", mean_baseline), ("revised mean", mean_revised)):
    line = mean_lines[label]
    assert line.get_visible() and _effective_alpha(line) > 0
    assert line.get_linestyle() == "--"
    assert np.allclose(line.get_xdata(), expected_mean, atol=1e-12, rtol=0)
legend_text = [item.get_text() for item in ax.get_legend().get_texts()]
assert set(legend_text) == {"baseline", "revised", "baseline mean", "revised mean"}
legend = ax.get_legend()
legend_handles = legend.legend_handles
assert legend.get_visible() and len(legend_handles) == 4
assert all(handle.get_visible() and _effective_alpha(handle) > 0 for handle in legend_handles)
assert all(item.get_visible() and _effective_alpha(item) > 0 for item in legend.get_texts())
plt.close(fig)

## Part C — justify the library boundary

In 4–6 sentences, explain why the distribution layers belong in `sns.histplot` while the exact mean markers belong in `ax.axvline`. State what the shared bins make comparable, what the summary numbers establish, and one limitation that the plot alone cannot resolve.

*Write your justification here.*

`sns.histplot` owns the distribution layers because it converts observations into binned counts and draws their bars. `ax.axvline` is the appropriate axes method for an exact scalar reference, so each mean is shown at its numerical location rather than being binned. Shared edges make corresponding bar heights comparable over identical score intervals, and the widened range ensures all 200 baseline and all 200 revised observations are counted. The summary floats establish both sample means and their signed revised-minus-baseline difference. The plot and summaries alone cannot establish causality or whether the observed shift will generalize to a new population.

### Answer check

The immutable assertions independently reproduce both seeded 200-point samples, verify the shared enclosing 13-bin edge array, intercept exactly two visible Seaborn layers, count all 400 observations, and check bar geometry, alpha, mean lines, legend, figure size, labels, and float summaries with `atol=1e-12` and `rtol=0`. The `plt.show()` sentinel also confirms that `build_comparison` does not display implicitly.